# Reinforce

Q-Learning 和 DNQ 更新的是价值函数 Q(s,a)（"这个动作值多少分"），策略是通过 argmaxQ 隐式得到的——先有分数表，再从中挑最好的。REINFORCE 直接更新策略参数 θ，跳过了 Q 值这一步——不问"值多少分"，直接学"该做什么"。

这个区别带来了两个关键后果：Q-Learning 是 off-policy 的（可以用旧数据反复训练），REINFORCE 是 on-policy 的（必须用当前策略的新数据）；Q-Learning 只能处理离散动作（需要遍历所有动作取 max），REINFORCE 可以处理连续动作（直接对概率密度求梯度）。

[Reinforce添加 log 的原因](https://walkinglabs.github.io/hands-on-modern-rl/chapter08_policy_gradient/reinforce#%E5%AF%B9%E6%95%B0%E5%AF%BC%E6%95%B0%E6%8A%80%E5%B7%A7)

 REINFORCE 算法中，策略梯度的估计公式为：

$$
\nabla_\theta J \approx \nabla_\theta \log \pi(a \mid s) \cdot G_t
$$

其中，$G_t$ 是从当前步到 episode 结束的总回报（即折扣累积回报）。

1. 存在的问题：高方差
$G_t$ 的波动极其巨大。即使在同一个策略、同一个状态下，运行两次也可能拿到完全不同的 $G_t$ 值，这会导致梯度估计的方差过高，训练不稳定。

1. 引入基线（Baseline）
为了降低方差，我们在梯度估计中减去一个基线 $V(s)$：

$$
\nabla_\theta J \approx \nabla_\theta \log \pi(a \mid s) \cdot (G_t - V(s))
$$

注：减去基线不会改变梯度的无偏性，但能显著降低方差。

下面基线是用来去方差的，常见的几种方法是：
1. 直接把 GT 进行平均求值
2. 使用一个价值网络进行预测，然后，这个网络的目标函数是不断逼近GT

In [ ]:
# 5. 从后往前倒推，计算每一步的累积回报 G_t
G_t_list = []
R = 0  # 游戏结束时的未来奖励是 0
gamma = 0.99 # 折扣因子，越远的奖励越打折扣

# reversed 把奖励列表反过来遍历
for r in reversed(rewards):
    R = r + gamma * R  # 当前步的实际回报 = 这一步的奖励 + 打折后的未来回报
    G_t_list.insert(0, R) # 塞到列表最前面，保证顺序和原来一致

G_t_tensor = torch.tensor(G_t_list)

# 2. 准备基线 V(s_t) 
# 假设你已经有了一个价值网络来预测每个状态的价值，或者用最简单的常数均值
baselines = ... # 这里代表算出的基线 (长度要和 G_t_tensor 一样)

# 3. 计算优势 A_t = G_t - 基线
advantages = G_t_tensor - baselines

# 6. 计算最终的 Loss 
policy_loss = []
for log_prob, A_t in zip(log_probs, advantages):
    # 核心公式完美复现： Loss = -log(pi(a|s)) * A_t
    policy_loss.append(-log_prob * A_t) 

# 把所有步的 Loss 加起来，求平均
loss = torch.stack(policy_loss).sum()

# 7. 神经网络反向传播老三样：清空梯度 -> 算梯度 -> 更新参数
optimizer.zero_grad()
loss.backward()
optimizer.step()

这里G_t 也可以换成TD error

# Actor-Critic 策略更新公式解析

## 核心公式
$$ \theta \leftarrow \theta + \alpha_\theta \cdot \delta \cdot \nabla_\theta \log \pi(\text{右}|S) $$

## 参数详解

- **$\theta$ (Theta)**：**策略网络的参数**。
  代表 Actor 神经网络中的权重（Weights）和偏置（Biases）。在更新过程中，$\theta$ 会不断迭代，使得策略网络越来越聪明。公式中的 $\leftarrow$ 表示将计算出的新值重新赋值给 $\theta$。

- **$\alpha_\theta$ (Alpha)**：**Actor 的学习率**。
  一个超参数，决定了每次参数更新的步长大小。它控制着神经网络参数更新的幅度，防止更新过大导致模型震荡，或更新过小导致学习过慢。

- **$\delta$ (Delta)**：**时序差分误差（TD Error）**。
  充当“评价信号”。它衡量了实际获得的奖励加上下一状态的估计价值，与当前状态价值预期之间的差距。
  - 当 $\delta > 0$ 时，说明该动作比预期好，更新会**增加**该动作的概率。
  - 当 $\delta < 0$ 时，说明该动作比预期差，更新会**降低**该动作的概率。

- **$\nabla_\theta \log \pi(\text{右}|S)$**：**对数概率的梯度（得分函数）**。
  - $\pi(\text{右}|S)$ 表示在状态 $S$ 下，策略网络输出选择动作“右”的概率。
  - $\nabla_\theta \log$ 代表对参数 $\theta$ 求梯度。
  它提供了“调整方向”，即：如果想增加“向右走”这个动作的概率，网络参数 $\theta$ 应该往哪个方向调整。